# Лабораторная работа 1 — рабочий ноутбук

**ФИО:** Таболин Владислав Андреевич

**Файлы данных:** clients.csv, transactions.csv (вариант S30)

Этот ноутбук — чистовой рабочий документ, а не черновик. Он должен
исполняться сверху вниз без ошибок на выданном вам **сыром** файле
и на выходе давать очищенную таблицу и `cleaning_log.csv`.

Разделы ниже обязательны, порядок менять не нужно. Внутри раздела
добавляйте столько ячеек, сколько нужно. Если какой-то раздел у вас
пустой — напишите в нём одной строкой, почему.

## 0. Окружение и настройки

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 50)

RAW_CLIENTS = "clients.csv"
RAW_TXN = "transactions.csv"

# Границы периодов из задания
DATA_START = pd.Timestamp("2024-01-01")
DATA_END = pd.Timestamp("2025-06-30 23:59:59")
BASE_START, BASE_END = pd.Timestamp("2024-01-01"), pd.Timestamp("2024-12-31 23:59:59")   # 12 мес.
EVAL_START, EVAL_END = pd.Timestamp("2025-01-01"), pd.Timestamp("2025-06-30 23:59:59")   # 6 мес.
SNAPSHOT = pd.Timestamp("2025-06-30")   # дата среза справочника клиентов

## 1. Загрузка сырых данных

Файлы выгружены разными системами. Убедитесь, что они прочитались
**правильно**: кодировка, разделитель, десятичный разделитель, типы,
даты, идентификаторы.

Сначала смотрим на первые байты файлов, чтобы определить кодировку и разделитель,
а не подбирать их вслепую.

In [ ]:
for path in [RAW_CLIENTS, RAW_TXN]:
    raw = open(path, "rb").read(300)
    for enc in ["utf-8", "cp1251"]:
        try:
            print(f"{path} как {enc}:", raw.decode(enc).splitlines()[1][:90])
        except UnicodeDecodeError:
            print(f"{path} как {enc}: не декодируется")
    print()

Вывод: `clients.csv` в UTF-8, разделитель `,`, десятичная точка.
`transactions.csv` в cp1251 (в UTF-8 не читается), разделитель `;`, десятичная запятая (`90,93`).
Все поля читаем как строки (`dtype=str`), чтобы pandas ничего не «исправил» молча
(например, не срезал ведущие нули в `branch_code`). Типы приводим явно в разделе 3.

In [ ]:
clients_raw = pd.read_csv(RAW_CLIENTS, sep=",", encoding="utf-8", dtype=str)
txn_raw = pd.read_csv(RAW_TXN, sep=";", encoding="cp1251", dtype=str)

print("clients:", clients_raw.shape, "| transactions:", txn_raw.shape)
display(clients_raw.head(3))
display(txn_raw.head(3))

## 2. Аудит качества

Здесь вы ничего не исправляете — только находите и измеряете.
Для каждой метрики качества заранее определите знаменатель.

### 2.1. Объём, типы, ключи, единица наблюдения

In [ ]:
# Единица наблюдения: clients — клиент (ожидаем уникальный client_id), transactions — операция (уникальный txn_id)
for name, df, key in [("clients", clients_raw, "client_id"), ("transactions", txn_raw, "txn_id")]:
    print(f"{name}: строк {len(df)}, столбцов {df.shape[1]}, уникальных {key}: {df[key].nunique()}")

audit = pd.DataFrame({
    "пропусков": clients_raw.isna().sum(),
    "доля пропусков, % от 5729": (clients_raw.isna().mean() * 100).round(1),
    "уникальных": clients_raw.nunique(),
})
audit

In [ ]:
pd.DataFrame({
    "пропусков": txn_raw.isna().sum(),
    "уникальных": txn_raw.nunique(),
})

### 2.2. Дубли

In [ ]:
n_c = len(clients_raw)
full_dup_c = clients_raw.duplicated().sum()
id_dup_c = clients_raw["client_id"].duplicated().sum()
print(f"clients: полных дублей строк {full_dup_c} из {n_c}; повторов client_id {id_dup_c}")

# Повторы client_id, которые НЕ являются полными дублями: чем различаются строки?
dup_ids = clients_raw[clients_raw["client_id"].duplicated(keep=False)].drop_duplicates()
diff = dup_ids.groupby("client_id").nunique()
diff = diff.loc[(diff > 1).any(axis=1)]
print("клиентов с расхождениями:", len(diff))
print("в каких полях расхождения:", (diff > 1).sum()[lambda s: s > 0].to_dict())

In [ ]:
n_t = len(txn_raw)
full_dup_t = txn_raw.duplicated().sum()
print(f"transactions: полных дублей {full_dup_t} из {n_t}")

# Повторы txn_id среди оставшихся строк
t_nodup = txn_raw.drop_duplicates()
same_id = t_nodup[t_nodup["txn_id"].duplicated(keep=False)].sort_values("txn_id")
print("строк с повторным txn_id после удаления полных дублей:", len(same_id))
# Отличаются ли они чем-то, кроме формата даты?
other_cols = [c for c in txn_raw.columns if c != "txn_datetime"]
only_dt = same_id.groupby("txn_id")[other_cols].nunique().le(1).all(axis=1)
print("пар, различающихся только полем txn_datetime:", only_dt.sum(), "из", same_id["txn_id"].nunique())
same_id.head(4)

### 2.3. Соответствие данных описанию полей из задания

In [ ]:
# Категориальные поля clients: какие значения на самом деле
for col in ["gender", "city", "segment", "loyalty_tier", "card_type", "income_source",
            "has_mobile_app", "marketing_opt_in", "satisfaction_score"]:
    print(col, "->", clients_raw[col].value_counts(dropna=False).to_dict(), "\n")

In [ ]:
# Категории операций: в описании «одна из девяти»
print("уникальных category:", txn_raw["category"].nunique())
print(txn_raw["category"].value_counts().to_dict(), "\n")
print("валюты:", txn_raw["currency"].value_counts().to_dict())
print("каналы:", txn_raw["channel"].value_counts().to_dict())
print("статусы:", txn_raw["status"].value_counts().to_dict())

In [ ]:
# category и mcc_code «соответствуют» друг другу? Таблица сопряжённости
pd.crosstab(txn_raw["category"], txn_raw["mcc_code"])

In [ ]:
# Числа и даты
inc = pd.to_numeric(clients_raw["monthly_income"])
print("monthly_income:", inc.describe().round(1).to_dict())
print("  значение -999:", (inc == -999).sum(), "| от 0 до 1000 руб.:", ((inc > 0) & (inc < 1000)).sum())

amt = pd.to_numeric(txn_raw["amount"].str.replace(",", ".", regex=False))
print("\namount:", amt.describe().round(2).to_dict())
print("  amount <= 0:", (amt <= 0).sum(), "| из них со статусом «возврат»:",
      ((amt <= 0) & (txn_raw["status"] == "возврат")).sum())
print("  медиана amount по валютам:", amt.groupby(txn_raw["currency"]).median().round(2).to_dict())

dt_len = txn_raw["txn_datetime"].str.len().value_counts().to_dict()
print("\nдлина строки txn_datetime (10 = только дата, 19 = дата и время):", dt_len)
dt = pd.to_datetime(txn_raw["txn_datetime"], format="mixed")
print("диапазон дат операций:", dt.min(), "—", dt.max())
print("вне периода 2024-01-01..2025-06-30:", ((dt < DATA_START) | (dt > DATA_END)).sum())

print("\nформаты дат в clients: birth_date", clients_raw["birth_date"].iloc[0],
      "| registration_date", clients_raw["registration_date"].iloc[0],
      "| app_last_login", clients_raw["app_last_login"].dropna().iloc[0])

In [ ]:
# Логические связи между полями
print(pd.crosstab(clients_raw["has_mobile_app"], clients_raw["app_last_login"].isna(),
                  rownames=["has_mobile_app"], colnames=["app_last_login пуст"]))
reg = pd.to_datetime(clients_raw["registration_date"], format="%d.%m.%Y")
card = pd.to_datetime(clients_raw["card_open_date"], format="%d.%m.%Y")
birth = pd.to_datetime(clients_raw["birth_date"])
print("\nкарта открыта раньше регистрации:", (card < reg).sum(),
      "| регистрация раньше рождения:", (reg < birth).sum())
age = (SNAPSHOT - birth).dt.days / 365.25
print("возраст на 30.06.2025: от", round(age.min(), 1), "до", round(age.max(), 1))

### 2.4. Связь между таблицами

In [ ]:
cl_ids = set(clients_raw["client_id"])
tx_ids = set(txn_raw["client_id"])
orphan = ~txn_raw["client_id"].isin(cl_ids)
print(f"клиентов в операциях: {len(tx_ids)}, в справочнике: {len(cl_ids)}")
print(f"клиентов в операциях, которых нет в справочнике: {len(tx_ids - cl_ids)} "
      f"({orphan.sum()} операций, {orphan.mean():.1%} от {len(txn_raw)})")
print(f"клиентов справочника без единой операции: {len(cl_ids - tx_ids)}")

# Когда у «потерянных» клиентов последняя операция?
last_orphan = pd.to_datetime(txn_raw.loc[orphan, "txn_datetime"], format="mixed") \
    .groupby(txn_raw.loc[orphan, "client_id"]).max()
print("\nпоследняя операция клиентов не из справочника, по месяцам:")
print(last_orphan.dt.to_period("M").value_counts().sort_index().to_dict())

**Выводы по аудиту.**

| # | что найдено | сколько | насколько мешает |
|---|---|---|---|
| 1 | разные кодировки и форматы чисел в двух файлах | 2 файла | без явных параметров чтения текст и суммы ломаются |
| 2 | полные дубли клиентов | 41 строка | клиент посчитается дважды |
| 3 | повтор `client_id` с разными значениями полей (`city` — написание, `segment` — по существу) | 14 клиентов | неясно, какой сегмент верный |
| 4 | полные дубли операций | 1307 строк | завышают активность |
| 5 | одна операция записана дважды: с временем и без | 193 пары | завышают активность |
| 6 | `txn_datetime` без времени | ≈8% строк, равномерно по каналам | для помесячной метрики не мешает |
| 7 | 12 написаний категорий вместо 9; часть `category` не совпадает с `mcc_code` | см. таблицу | искажает разрезы по категориям |
| 8 | 17 написаний 6 городов | все строки | искажает разрезы по городам |
| 9 | `monthly_income`: код `-999` и значения в тысячах рублей | 52 и 130 | искажают доход |
| 10 | операции вне периода данных (2023 г., после 06.2025) | см. вывод | вне окон расчёта |
| 11 | клиенты в операциях, которых нет в справочнике действующих | 326 клиентов | ушедшие клиенты, к ним кампанию не обращаем |
| 12 | валюты USD/EUR | ≈0,4% операций | суммы в разных единицах |
| 13 | статусы «отклонено» и «возврат» | ≈3% операций | это не покупки клиента |
| 14 | `satisfaction_score` пуст у 59% клиентов | 3387 | опрос проходили не все |

Расхождения с описанием от владельца данных: категорий 12, а не 9; в поле `city` много вариантов
написания; `mcc_code` не всегда соответствует категории; доход частично в тысячах рублей;
в операциях есть клиенты, которых нет в справочнике; есть даты вне заявленного периода.

## 3. Препроцессинг

На каждое изменение — строка в журнале (раздел 4). Решение «оставить
как есть» тоже является решением и тоже попадает в журнал.

In [ ]:
# Журнал решений: одна строка — одно решение. В разделе 4 превращается в DataFrame.
log_rows = []

def log(field, issue, scope, n_rows, value_before, action, value_after, rationale):
    log_rows.append(dict(field=field, issue=issue, scope=scope, n_rows=int(n_rows),
                         value_before=str(value_before), action=action,
                         value_after=str(value_after), rationale=rationale))

clients = clients_raw.copy()
txn = txn_raw.copy()

### 3.1. Типы, даты, числа

In [ ]:
# --- Дубли клиентов ---
n0 = len(clients)
clients = clients.drop_duplicates()
log("все поля", "полные дубли строк", "дублирующиеся строки clients", n0 - len(clients),
    "строка повторена", "удалено", "одна строка на клиента",
    "строки идентичны, повтор — артефакт выгрузки")

# --- Дубли операций ---
n0 = len(txn)
txn = txn.drop_duplicates()
log("все поля", "полные дубли операций", "дублирующиеся строки transactions", n0 - len(txn),
    "операция повторена", "удалено", "одна строка на операцию",
    "одинаковые txn_id и все поля — одна и та же операция, иначе активность завышена")

# Пары «с временем / без времени»: оставляем запись с временем (она полнее)
txn["_has_time"] = txn["txn_datetime"].str.len() > 10
txn = txn.sort_values(["txn_id", "_has_time"], ascending=[True, False])
n0 = len(txn)
dup_pair_ids = txn.loc[txn["txn_id"].duplicated(), "txn_id"].tolist()
txn = txn.drop_duplicates("txn_id", keep="first")
log("txn_datetime", "одна операция записана дважды: с временем и без", "повторные txn_id",
    n0 - len(txn), "2024-01-04 / 2024-01-04 15:26:09", "удалено", "оставлена запись с временем",
    "кроме формата даты строки совпадают, это одна операция")

# --- Даты ---
txn["txn_datetime"] = pd.to_datetime(txn["txn_datetime"], format="mixed")
n_no_time = (~txn["_has_time"]).sum()
log("txn_datetime", "у части операций нет времени", "строки длиной 10 символов", n_no_time,
    "2023-11-27", "оставлено", "время 00:00:00",
    "метрика помесячная, время не нужно; доля одинакова во всех каналах — не связано с каналом")
txn = txn.drop(columns="_has_time")

clients["birth_date"] = pd.to_datetime(clients["birth_date"], format="%Y-%m-%d")
clients["registration_date"] = pd.to_datetime(clients["registration_date"], format="%d.%m.%Y")
clients["card_open_date"] = pd.to_datetime(clients["card_open_date"], format="%d.%m.%Y")
clients["app_last_login"] = pd.PeriodIndex(clients["app_last_login"], freq="M")
log("registration_date, card_open_date", "формат ДД.ММ.ГГГГ, отличный от birth_date",
    "все строки", len(clients), "02.04.2014", "исправлено", "2014-04-02",
    "приведение к единому типу даты, смысл не меняется")

# --- Числа ---
txn["amount"] = pd.to_numeric(txn["amount"].str.replace(",", ".", regex=False))
txn["mcc_code"] = txn["mcc_code"].astype(int)
log("amount", "десятичная запятая", "все строки transactions", len(txn), "90,93",
    "исправлено", "90.93", "формат выгрузки cp1251-системы, смысл не меняется")

clients["monthly_income"] = pd.to_numeric(clients["monthly_income"])
m = clients["monthly_income"] == -999
log("monthly_income", "код -999 вместо значения", "monthly_income == -999", m.sum(), -999,
    "исправлено", "NaN", "отрицательного дохода не бывает, это код «нет данных» в системе-источнике")
clients.loc[m, "monthly_income"] = np.nan

m = (clients["monthly_income"] > 0) & (clients["monthly_income"] < 1000)
before = clients.loc[m, "monthly_income"].median()
log("monthly_income", "значения в тысячах рублей", "0 < monthly_income < 1000", m.sum(),
    f"медиана {before:.1f}", "исправлено", f"медиана {before * 1000:.0f}",
    "доход меньше 1000 руб./мес. неправдоподобен; после умножения на 1000 значения в диапазоне остальных клиентов")
clients.loc[m, "monthly_income"] *= 1000

clients["satisfaction_score"] = pd.to_numeric(clients["satisfaction_score"]).astype("Int64")
clients["has_mobile_app"] = clients["has_mobile_app"].map({"Да": True, "Нет": False})
clients["marketing_opt_in"] = clients["marketing_opt_in"].map({"Да": True, "Нет": False})

### 3.2. Категориальные признаки

In [ ]:
# --- Города: 17 написаний -> 6 городов ---
def norm_city(s):
    s = s.strip().lower().replace("г.", "").strip()
    return {"москва": "Москва", "казань": "Казань", "санкт-петербург": "Санкт-Петербург",
            "санкт петербург": "Санкт-Петербург", "спб": "Санкт-Петербург",
            "нижний новгород": "Нижний Новгород", "н. новгород": "Нижний Новгород",
            "екатеринбург": "Екатеринбург", "новосибирск": "Новосибирск"}[s]

before_n = clients["city"].nunique()
changed = (clients["city"] != clients["city"].map(norm_city)).sum()
clients["city"] = clients["city"].map(norm_city)
log("city", "разнобой написания", "все строки", changed,
    f"{before_n} вариантов, напр. «г. Москва», « москва», «СПб»", "исправлено",
    f"{clients['city'].nunique()} городов", "регистр, пробелы, «г.», сокращения; смысл не меняется")

# --- Повтор client_id с разным сегментом ---
dup = clients[clients["client_id"].duplicated(keep=False)]
n_conf = dup["client_id"].nunique()
print("клиентов с двумя записями после нормализации города:", n_conf)
print(dup.groupby("client_id").nunique().gt(1).sum()[lambda s: s > 0].to_dict())
# Источника истины нет. Оставляем последнюю запись как более позднюю версию справочника.
conflict_ids = dup["client_id"].unique().tolist()
clients = clients.drop_duplicates("client_id", keep="last")
log("segment", "у клиента две записи с разным сегментом", "client_id: " + ", ".join(conflict_ids),
    n_conf, "напр. Массовый / Премиальный", "удалено", "оставлена последняя запись",
    "источника истины нет; предполагаем, что выгрузка упорядочена по времени. Клиентов мало, на выводы не влияет")

# --- Категории операций: категорию берём по mcc_code ---
mcc_to_cat = {5411: "Продукты", 4111: "Транспорт", 5651: "Одежда и обувь", 5541: "АЗС",
              7832: "Развлечения и досуг", 5812: "Кафе и рестораны", 5912: "Аптеки",
              5815: "Онлайн-сервисы", 4814: "Связь и интернет"}
cat_from_mcc = txn["mcc_code"].map(mcc_to_cat)
spelling = {"Связь и Интернет": "Связь и интернет", "кафе и рестораны": "Кафе и рестораны",
            "Кафе/рестораны": "Кафе и рестораны"}
cat_norm = txn["category"].replace(spelling)

n_spell = (txn["category"] != cat_norm).sum()
log("category", "разнобой написания", "3 варианта написания", n_spell,
    "«Связь и Интернет», «кафе и рестораны», «Кафе/рестораны»", "исправлено",
    "9 категорий", "регистр и разделитель; смысл не меняется")

n_mismatch = (cat_norm != cat_from_mcc).sum()
log("category", "категория не соответствует mcc_code", "category != категория по mcc_code", n_mismatch,
    "напр. «Продукты» при mcc 4111", "исправлено", "категория по mcc_code",
    "mcc_code — код из платёжной системы, присваивается автоматически; расхождения малы (<1%) и разбросаны случайно")
txn["category"] = cat_from_mcc
print(txn["category"].value_counts())

### 3.3. Пропуски

Для каждого поля с пропусками: доля, предположение о механизме, решение.

In [ ]:
na = clients.isna().sum()
na = na[na > 0]
pd.DataFrame({"пропусков": na, "доля, % от числа клиентов": (na / len(clients) * 100).round(1)})

In [ ]:
# monthly_income: от чего зависит пропуск?
print(pd.crosstab(clients["income_source"], clients["monthly_income"].isna(), normalize="index").round(3))
# satisfaction_score: от чего зависит пропуск?
print(pd.crosstab(clients["segment"], clients["satisfaction_score"].isna(), normalize="index").round(3))

**Механизмы пропусков и решения.**

- `app_last_login` пуст ровно у клиентов без приложения (`has_mobile_app = Нет`). Это не пропуск,
  а отсутствие события: входа не было. Оставляем.
- `monthly_income` (с учётом бывших `-999`): пропуск заметно чаще у клиентов с `income_source = Не указан`.
  Это MAR: пропуск зависит от наблюдаемого поля, клиент не сообщил источник и доход.
  Доход не входит в метрику и в правила групп, поэтому **не заполняем**: заполнение нарушило бы
  распределение, а пользы для задачи нет.
- `satisfaction_score` пуст у большинства: опрос проходили не все, доля пропусков различается по сегментам.
  Скорее всего, MNAR или отбор: отвечают те, кому есть что сказать. **Не заполняем**, используем
  только как дополнительный разрез с указанием n. Шкала порядковая, поэтому центр — медиана, не среднее.

In [ ]:
log("monthly_income", "пропуски (с учётом бывших -999)", "monthly_income is NaN",
    clients["monthly_income"].isna().sum(), "NaN", "оставлено", "NaN",
    "MAR: связано с income_source = Не указан; доход не входит в метрику и правила групп")
log("satisfaction_score", "пропуски", "satisfaction_score is NaN",
    clients["satisfaction_score"].isna().sum(), "NaN", "оставлено", "NaN",
    "опрос проходили не все (отбор); порядковая шкала, заполнение средним недопустимо")

### 3.4. Выбросы

Обнаружение и решение — это два разных шага. Сначала первый, потом второй.

**Шаг 1. Обнаружение.**

In [ ]:
ok_rub = txn[(txn["status"] == "успешно") & (txn["currency"] == "RUB")]["amount"]
q1, q3 = ok_rub.quantile([0.25, 0.75])
upper = q3 + 1.5 * (q3 - q1)
print(f"amount, успешные RUB: n = {len(ok_rub)}, медиана = {ok_rub.median():.0f} руб.")
print(f"граница МКР (Q3 + 1.5·IQR) = {upper:.0f} руб.; выше неё: {(ok_rub > upper).sum()} операций "
      f"({(ok_rub > upper).mean():.1%})")
print("99.9-й перцентиль:", round(ok_rub.quantile(0.999)), "руб.; максимум:", round(ok_rub.max()), "руб.")

fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].hist(ok_rub, bins=100)
ax[0].set(title="Суммы успешных операций, RUB", xlabel="сумма, руб.", ylabel="число операций")
sns.boxplot(x=txn.loc[ok_rub.index, "category"], y=ok_rub, ax=ax[1], showfliers=True)
ax[1].set(title="По категориям", xlabel="", ylabel="сумма, руб.")
ax[1].tick_params(axis="x", rotation=60)
plt.tight_layout(); plt.show()

big = txn.loc[ok_rub[ok_rub > 50000].index]
print("операции дороже 50 000 руб. по категориям:", big["category"].value_counts().to_dict())

**Шаг 2. Решение.**

Распределение сумм сильно скошено вправо, поэтому граница МКР отсекает около 8% операций.
Это не ошибки, а обычные крупные покупки. Самые крупные суммы (свыше 50 000 руб.) приходятся
на «Одежду и обувь»: пальто или техника за такую цену правдоподобны. Источник не позволяет
проверить их иначе, признаков ошибки ввода (лишние нули, отрицательные значения у успешных)
нет. **Оставляем все.** Основная метрика активности — число операций, на неё выбросы
по сумме не влияют.

Отрицательные суммы есть только у статуса «возврат»: это не выброс, а смысл статуса.
Медиана сумм в USD/EUR около 11 против ≈1070 в RUB: суммы указаны в валюте операции.
Курса в данных нет, поэтому суммы в валюте не пересчитываем, а в суммовых разрезах берём только RUB.

In [ ]:
log("amount", "суммы выше границы МКР", f"успешные RUB, amount > {upper:.0f}", (ok_rub > upper).sum(),
    f"> {upper:.0f} руб., макс. {ok_rub.max():.0f}", "оставлено", "без изменений",
    "скошенное распределение крупных покупок, признаков ошибки нет; основная метрика — число операций")
log("amount, currency", "суммы в USD/EUR в валюте операции", "currency in (USD, EUR)",
    (txn["currency"] != "RUB").sum(), "медиана ≈11", "оставлено", "без пересчёта",
    "курса в данных нет; в суммовых разрезах берём только RUB, в счёте операций — все")

### 3.5. Фильтры, применяемые перед расчётом активности

Покажите воронку: сколько строк на входе и сколько осталось после каждого фильтра.

In [ ]:
funnel = [("после удаления дублей", len(txn), txn["client_id"].nunique())]

in_period = txn["txn_datetime"].between(DATA_START, DATA_END)
log("txn_datetime", "операции вне периода данных", "дата < 2024-01-01 или > 2025-06-30",
    (~in_period).sum(), f"{txn['txn_datetime'].min():%Y-%m-%d} … {txn['txn_datetime'].max():%Y-%m-%d}",
    "удалено", "2024-01-01 … 2025-06-30", "вне заявленного периода и окон расчёта; единичные строки — ошибки дат")
txn = txn[in_period]
funnel.append(("в периоде 2024-01 … 2025-06", len(txn), txn["client_id"].nunique()))

known = txn["client_id"].isin(clients["client_id"])
log("client_id", "клиента нет в справочнике действующих клиентов",
    "client_id not in clients", (~known).sum(),
    f"{txn.loc[~known, 'client_id'].nunique()} клиентов", "удалено", "исключены из расчёта",
    "справочник — действующие клиенты на 30.06.2025; остальные ушли из банка, кампанией их не охватить")
orphans = txn.loc[~known, ["client_id", "txn_datetime"]].copy()   # пригодится в разделе 7.1
txn = txn[known]
funnel.append(("клиент есть в справочнике", len(txn), txn["client_id"].nunique()))

ok = txn["status"] == "успешно"
txn_ok = txn[ok].copy()
funnel.append(("статус «успешно»", len(txn_ok), txn_ok["client_id"].nunique()))

pd.DataFrame(funnel, columns=["шаг", "операций", "клиентов"])

## 4. Журнал изменений

Одна строка — одно решение, а не одна запись. Колонки: `field`, `issue`,
`scope`, `n_rows`, `value_before`, `action`, `value_after`, `rationale`.

In [ ]:
cleaning_log = pd.DataFrame(log_rows, columns=[
    "field", "issue", "scope", "n_rows",
    "value_before", "action", "value_after", "rationale",
])
cleaning_log

## 5. Разведочный анализ

Графиков столько, сколько нужно. В отчёт пойдут не все.

### 5.1. Распределения количественных признаков

In [ ]:
# Подготовка: атрибуты клиента к операциям, месяц и окно
clients["age"] = ((SNAPSHOT - clients["birth_date"]).dt.days / 365.25).round(1)
txn_ok = txn_ok.merge(clients[["client_id", "segment", "city", "has_mobile_app", "loyalty_tier",
                               "card_type", "income_source", "age"]], on="client_id", how="left")
txn_ok["month"] = txn_ok["txn_datetime"].dt.to_period("M")
txn_ok["window"] = np.where(txn_ok["txn_datetime"] <= BASE_END, "база 2024", "оценка 2025")
MONTHS = pd.period_range("2024-01", "2025-06", freq="M")

fig, ax = plt.subplots(2, 2, figsize=(12, 7))
ax[0, 0].hist(clients["age"], bins=30)
ax[0, 0].set(title=f"Возраст клиентов, n = {len(clients)}", xlabel="лет на 30.06.2025", ylabel="клиентов")
inc = clients["monthly_income"].dropna()
ax[0, 1].hist(inc, bins=60)
ax[0, 1].set(title=f"Доход, n = {len(inc)} (без пропусков)", xlabel="руб./мес.", ylabel="клиентов")
ops_per_client = txn_ok.groupby("client_id").size()
ax[1, 0].hist(ops_per_client, bins=60)
ax[1, 0].set(title=f"Успешных операций на клиента за 18 мес., n = {len(ops_per_client)}",
             xlabel="операций", ylabel="клиентов")
rub = txn_ok.loc[txn_ok["currency"] == "RUB", "amount"]
ax[1, 1].hist(np.log10(rub.clip(lower=1)), bins=60)
ax[1, 1].set(title=f"Сумма операции, RUB, n = {len(rub)}", xlabel="log10(сумма, руб.)", ylabel="операций")
plt.tight_layout(); plt.show()

desc = pd.DataFrame({
    "медиана": [clients["age"].median(), inc.median(), ops_per_client.median(), rub.median()],
    "Q1": [clients["age"].quantile(.25), inc.quantile(.25), ops_per_client.quantile(.25), rub.quantile(.25)],
    "Q3": [clients["age"].quantile(.75), inc.quantile(.75), ops_per_client.quantile(.75), rub.quantile(.75)],
    "среднее": [clients["age"].mean(), inc.mean(), ops_per_client.mean(), rub.mean()],
}, index=["возраст, лет", "доход, руб./мес.", "операций на клиента", "сумма операции, руб."]).round(1)
desc

**Выбор мер.** Доход, число операций и суммы сильно скошены вправо (длинный хвост, среднее заметно
выше медианы), поэтому центр описываем **медианой**, разброс **межквартильным размахом** (Q1–Q3).
Возраст распределён почти симметрично, среднее и медиана близки, но для единообразия тоже даём медиану.
`satisfaction_score` — порядковая шкала: для неё только медиана и частоты, среднее не считаем.

### 5.2. Частотные характеристики категориальных признаков

In [ ]:
cat_cols = ["segment", "loyalty_tier", "card_type", "city", "income_source", "has_mobile_app",
            "marketing_opt_in", "gender"]
for col in cat_cols:
    vc = clients[col].value_counts()
    print(f"{col} (n = {len(clients)}):", {k: f"{v} ({v / len(clients):.0%})" for k, v in vc.items()})
print("\nsatisfaction_score (n ответивших =", clients["satisfaction_score"].notna().sum(), "):",
      clients["satisfaction_score"].value_counts().sort_index().to_dict(),
      "| медиана:", clients["satisfaction_score"].median())

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
n_ops = len(txn_ok)
(txn_ok["category"].value_counts(normalize=True) * 100).sort_values().plot.barh(ax=ax[0])
ax[0].set(title=f"Категории, % от {n_ops} успешных операций", xlabel="% операций")
(txn_ok["channel"].value_counts(normalize=True) * 100).sort_values().plot.barh(ax=ax[1])
ax[1].set(title=f"Каналы, % от {n_ops} успешных операций", xlabel="% операций")
plt.tight_layout(); plt.show()
print(pd.crosstab(clients["city"], clients["segment"], normalize="index").round(2))

Половина клиентов — массовый сегмент. Состав городов неодинаков: в Казани **87% клиентов массового
сегмента**, в остальных городах около 45%. Это важно для разрезов по городам (см. раздел 8).

### 5.3. Динамика по месяцам

In [ ]:
monthly = txn_ok.groupby("month").size().reindex(MONTHS)
fig, ax = plt.subplots(figsize=(11, 3.5))
ax.plot(monthly.index.astype(str), monthly.values, marker="o")
ax.axvline(11.5, color="grey", ls="--")
ax.set(title="Успешные операции по месяцам, все клиенты справочника", ylabel="операций в месяц", xlabel="месяц")
ax.tick_params(axis="x", rotation=60)
plt.tight_layout(); plt.show()
print("в среднем за месяц: 2024 —", round(monthly[:12].mean()), "| янв–июн 2024 —", round(monthly[:6].mean()),
      "| янв–июн 2025 —", round(monthly[12:].mean()))

In [ ]:
def index_by(col, data=txn_ok):
    """Число операций по месяцам, деленное на среднемесячное за 2024 (1.0 = уровень 2024)."""
    p = data.pivot_table(index="month", columns=col, values="txn_id", aggfunc="count").reindex(MONTHS)
    return p / p.iloc[:12].mean()

fig, ax = plt.subplots(2, 2, figsize=(13, 7), sharex=True)
for a, col in zip(ax.flat, ["segment", "channel", "category", "has_mobile_app"]):
    idx = index_by(col)
    idx.index = idx.index.astype(str)
    idx.plot(ax=a, legend=True)
    a.axvline(11.5, color="grey", ls="--")
    a.set(title=f"Индекс операций по {col}", ylabel="к среднему за 2024", xlabel="месяц")
    a.legend(fontsize=7)
plt.tight_layout(); plt.show()

**Что видно в динамике.**
- **Сезонность.** Пик в декабре, провал в январе. Первое полугодие и в 2024 году было ниже среднего за год
  (≈93% от среднемесячного), поэтому часть «падения» в окне январь–июнь 2025 — сезонная.
- **Массовый сегмент** с января 2025 года просел на 30–40%, остальные сегменты почти на уровне 2024 года.
- **Каналы.** Операции по пластиковой карте падают, по **токену** (Pay-сервисы) растут на 30–45%.
  Клиенты меняют способ оплаты, а не перестают платить.
- **«Онлайн-банк»** провалился в апреле–мае 2025 года и вернулся в июне. Это похоже на технический сбой,
  а не на поведение клиентов.
- **Категории.** «Развлечения и досуг» обвалились с января 2025 года резко и во всех сегментах,
  остальные категории следуют общему тренду.

### 5.4. Разрезы и связи между признаками

In [ ]:
def change(data):
    """Изменение среднемесячного числа операций: янв–июн 2025 к янв–июн 2024 (без сезонности)."""
    h1_24 = data["month"].between(pd.Period("2024-01", "M"), pd.Period("2024-06", "M")).sum()
    h1_25 = (data["window"] == "оценка 2025").sum()
    return h1_25 / h1_24 - 1

cuts = {}
for col in ["segment", "has_mobile_app", "city", "card_type", "loyalty_tier", "income_source"]:
    cuts[col] = txn_ok.groupby(col).apply(change, include_groups=False).round(3)
for col, s in cuts.items():
    print(col, s.to_dict())

In [ ]:
seg_app = txn_ok.groupby(["segment", "has_mobile_app"]).apply(change, include_groups=False).unstack()
seg_city = txn_ok.groupby(["segment", "city"]).apply(change, include_groups=False).unstack()
seg_cat = txn_ok.groupby(["segment", "category"]).apply(change, include_groups=False).unstack()
fig, ax = plt.subplots(1, 3, figsize=(17, 4))
for a, tab, title in [(ax[0], seg_app, "сегмент × приложение"), (ax[1], seg_city, "сегмент × город"),
                      (ax[2], seg_cat, "сегмент × категория")]:
    sns.heatmap(tab * 100, annot=True, fmt=".0f", cmap="RdBu", center=0, ax=a,
                cbar_kws={"label": "% изменения, янв–июн 2025 к янв–июн 2024"})
    a.set(title=title, xlabel="", ylabel="")
plt.tight_layout(); plt.show()

**Связи.**
- Главный фактор — **сегмент × приложение**: массовый без приложения теряет больше всего (≈−40%),
  массовый с приложением около −20%, остальные сегменты почти без изменений.
- **Город**: внутри одного сегмента Казань не отличается от других городов. Общий провал Казани объясняется
  тем, что там почти одни массовые клиенты (эффект состава, см. раздел 8).
- **Развлечения** падают во всех сегментах, это отдельная от сегмента причина.

## 6. Метрика активности

Что именно вы считаете активностью, как она вычисляется, на каком
множестве клиентов, за какие окна. Отдельно: какое снижение эта метрика
**не увидит**.

**Метрика:** среднее **число успешных карточных операций в месяц** (все каналы: карта, токен, СБП,
онлайн-банк).

- **Почему число, а не сумма.** Заказчик говорит «стали реже пользоваться»: это про частоту.
  На сумму влияют разовые крупные покупки (выбросы мы оставили) и валютные операции без курса.
- **Почему все каналы.** Переход с пластика на токен — та же карта, другим способом.
  Если считать только «Карту», такие клиенты ложно выглядели бы «падающими».
- **Окна разной длины.** База — 2024 год, оценка — январь–июнь 2025 года. Сравниваем **в расчёте на месяц**.
- **Экспозиция.** Операций до открытия текущей карты не бывает (проверено ниже), поэтому базу клиента считаем
  с месяца **после** `card_open_date` (первый месяц неполный). Клиенты с картой, открытой после 01.12.2024,
  базы не имеют, и снижение у них не определено.
- **Изменение:** `chg = rate_2025 / rate_2024 − 1`. Падением считаем `chg ≤ −30%`. Это втрое больше
  сезонного эффекта (первое полугодие ≈ −7% к среднему за год), то есть отсекает сезонность с запасом.
- Клиенты с базой меньше 1 операции в месяц не оцениваются: при 0–5 операциях в год процент изменения шумный.

**Чего метрика не увидит:**
1. Снижения **суммы** трат при той же частоте (клиент так же часто платит, но на мелкие суммы,
   а крупные покупки ушли в другой банк).
2. Падения **внутри окна**: если клиент активен в январе–апреле и пропал в июне, среднее за 6 месяцев сгладит это
   (частично ловим отдельным признаком «конец окна»).
3. Перераспределения **между категориями** при той же частоте (ушёл из развлечений, но больше платит в продуктах).
4. Ухода тех, кто **уже закрыл карту**: их нет в справочнике, метрика по ним не считается (см. 7.1).

In [ ]:
first_txn = txn_ok.groupby("client_id")["txn_datetime"].min()
lag = (first_txn - clients.set_index("client_id")["card_open_date"]).dt.days
print("операций раньше открытия карты:", (lag < 0).sum(), "| медиана дней от открытия карты до первой операции в данных:", lag.median())

cl = clients.set_index("client_id")
M = (txn_ok.groupby(["client_id", "month"]).size().unstack(fill_value=0)
     .reindex(index=cl.index, columns=MONTHS, fill_value=0))
base_start = (cl["card_open_date"].dt.to_period("M") + 1).clip(lower=pd.Period("2024-01", "M"))
in_base = pd.DataFrame({m: m >= base_start for m in MONTHS[:12]})
n_base_months = in_base.sum(axis=1)

metrics = pd.DataFrame(index=cl.index)
metrics["n_base_months"] = n_base_months
metrics["rate_base"] = (M[MONTHS[:12]] * in_base).sum(axis=1) / n_base_months.replace(0, np.nan)
metrics["rate_eval"] = M[MONTHS[12:]].sum(axis=1) / 6
metrics["rate_eval_end"] = M[MONTHS[15:]].sum(axis=1) / 3          # апрель–июнь 2025
metrics["chg"] = metrics["rate_eval"] / metrics["rate_base"] - 1
metrics["chg_end"] = metrics["rate_eval_end"] / metrics["rate_base"] - 1

# то же по категории «Развлечения и досуг» и по остальным категориям
ent = txn_ok["category"] == "Развлечения и досуг"
ME = (txn_ok[ent].groupby(["client_id", "month"]).size().unstack(fill_value=0)
      .reindex(index=cl.index, columns=MONTHS, fill_value=0))
metrics["ent_base"] = (ME[MONTHS[:12]] * in_base).sum(axis=1) / n_base_months.replace(0, np.nan)
metrics["ent_eval"] = ME[MONTHS[12:]].sum(axis=1) / 6
metrics["ent_chg"] = metrics["ent_eval"] / metrics["ent_base"] - 1
metrics["other_chg"] = ((metrics["rate_eval"] - metrics["ent_eval"])
                        / (metrics["rate_base"] - metrics["ent_base"]) - 1)

metrics["measurable"] = (n_base_months >= 1) & (metrics["rate_base"] >= 1)
metrics = metrics.join(cl[["segment", "city", "has_mobile_app", "loyalty_tier", "card_type",
                           "income_source", "age", "marketing_opt_in", "card_open_date", "satisfaction_score"]])
m_ok = metrics[metrics["measurable"]]
print(f"клиентов в справочнике: {len(metrics)}; с измеримой метрикой: {len(m_ok)}")
print(f"медиана chg: {m_ok['chg'].median():.1%}; доля с chg ≤ −30%: {(m_ok['chg'] <= -0.3).mean():.1%} "
      f"(n = {(m_ok['chg'] <= -0.3).sum()} из {len(m_ok)})")

full = metrics["n_base_months"] == 12
season = (M.loc[full, MONTHS[:6]].sum().sum() / 6) / (M.loc[full, MONTHS[:12]].sum().sum() / 12)
print(f"сезонный коэффициент янв–июн к среднему за 2024 (клиенты с полным годом): {season:.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.5))
ax.hist(m_ok["chg"].clip(upper=2) * 100, bins=60)
ax.axvline(-30, color="red", ls="--", label="порог падения −30%")
ax.axvline((season - 1) * 100, color="grey", ls=":", label="сезонный эффект")
ax.set(title=f"Изменение активности клиента, n = {len(m_ok)}",
       xlabel="% изменения операций в месяц, 2025 к 2024 (обрезано на +200%)", ylabel="клиентов")
ax.legend(); plt.tight_layout(); plt.show()

## 7. Поиск и описание групп

Правило отбора для каждой группы, её размер, причина падения.
Правило разрешения конфликтов, если клиент подходит под несколько групп.

**Правила групп** (кто + что произошло). Порядок — приоритет: клиент, подходящий под несколько правил,
попадает в первую подходящую группу. Сначала идёт самая конкретная причина (одна категория),
потом сегментные.

| группа | кто | что произошло |
|---|---|---|
| **G1. Ушли из развлечений** | клиенты, регулярно платившие за «Развлечения и досуг» (в 2024 ≥ 0,5 операции/мес.) | с января 2025 траты на развлечения упали на 70% и больше, остальные траты сократились не более чем на 30% |
| **G2. Массовый без приложения: угасание** | массовый сегмент, нет мобильного приложения | активность упала на 30% и больше, к апрелю–июню опустилась примерно до трети прежней |
| **G3. Массовый с приложением: снижение** | массовый сегмент, приложение есть | активность упала на 30% и больше, равномерно по категориям |

Во все группы — только клиенты с согласием на коммуникацию и с измеримой метрикой.

In [ ]:
eligible = m_ok["marketing_opt_in"]
rules = {
    "G1_развлечения": (m_ok["ent_base"] >= 0.5) & (m_ok["ent_chg"] <= -0.7) & (m_ok["other_chg"] > -0.3),
    "G2_массовый_без_приложения": (m_ok["segment"] == "Массовый") & ~m_ok["has_mobile_app"] & (m_ok["chg"] <= -0.3),
    "G3_массовый_с_приложением": (m_ok["segment"] == "Массовый") & m_ok["has_mobile_app"] & (m_ok["chg"] <= -0.3),
}
# пересечения правил (до учёта согласия)
overlap = pd.DataFrame({a: [(ra & rb).sum() for rb in rules.values()] for a, ra in rules.items()}, index=rules.keys())
print("пересечения правил, клиентов:"); print(overlap)

m_ok = m_ok.copy()
m_ok["group"] = pd.NA
for g, r in rules.items():
    m_ok.loc[r & m_ok["group"].isna(), "group"] = g

campaign = m_ok[m_ok["group"].notna() & eligible].reset_index()[["client_id", "group"]]
print("\nразмер групп в кампании:"); print(campaign["group"].value_counts())
print("итого клиентов:", campaign["client_id"].nunique(), "| уникальны:", campaign["client_id"].is_unique)
assert campaign["client_id"].nunique() <= 2000
assert campaign["group"].value_counts().min() >= 150

In [ ]:
gm = m_ok[m_ok["group"].notna() & eligible]
summary = gm.groupby("group").agg(
    клиентов=("chg", "size"),
    медиана_операций_2024=("rate_base", "median"),
    медиана_операций_2025=("rate_eval", "median"),
    медиана_изменения=("chg", "median"),
    медиана_изм_развлечения=("ent_chg", "median"),
    медиана_изм_остальное=("other_chg", "median"),
    доля_с_приложением=("has_mobile_app", "mean"),
    медиана_возраста=("age", "median"),
).round(2)
summary

In [ ]:
# Траектория групп по месяцам: операции группы к её среднемесячному уровню 2024
fig, ax = plt.subplots(figsize=(11, 4))
for g in sorted(gm["group"].unique()):
    ids = gm.index[gm["group"] == g]
    traj = M.loc[ids].sum() / m_ok.loc[ids, "rate_base"].sum()
    ax.plot(MONTHS.astype(str), traj.values, marker="o", label=f"{g} (n = {len(ids)})")
rest = m_ok.index[m_ok["group"].isna()]
ax.plot(MONTHS.astype(str), (M.loc[rest].sum() / m_ok.loc[rest, "rate_base"].sum()).values,
        color="grey", ls="--", label=f"остальные измеримые (n = {len(rest)})")
ax.axvline(11.5, color="grey", ls=":")
ax.set(title="Активность групп по месяцам", ylabel="операций к среднему уровню 2024", xlabel="месяц")
ax.tick_params(axis="x", rotation=60); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

**Причины падения по группам.**
- **G1.** Активность в целом сохранилась, исчезла одна категория. Обвал развлечений в январе 2025 года
  одновременный и во всех сегментах и городах. Похоже на внешнюю причину в категории (условия кэшбэка
  или партнёрская сеть), а не на уход клиента. Предложение: вернуть выгоду в развлечениях.
- **G2.** Падение по всем категориям, к концу окна около трети прежней активности: клиент уходит из банка.
  Это тот же профиль, что у 326 клиентов, уже пропавших из справочника. Без приложения банку нечем удерживать
  клиента, кроме карты. Предложение: перевод в приложение и персональное предложение по карте.
- **G3.** Снижение умеренное и ровное, канал связи есть (приложение). Предложение: push-кампания
  с кэшбэком в основных категориях.

### 7.1. Кого в кампанию не берём и почему

In [ ]:
excl = []
excl.append(("ушли из банка: есть операции, нет в справочнике действующих", orphans["client_id"].nunique(),
             "карты закрыты, коммуникация как с действующими невозможна"))
new_card = (metrics["n_base_months"] == 0).sum()
excl.append(("карта открыта после 01.12.2024: нет базового периода", new_card,
             "снижение не определено, это новые клиенты"))
low = ((metrics["n_base_months"] >= 1) & (metrics["rate_base"] < 1)).sum()
excl.append(("базовая активность меньше 1 операции/мес.", low, "процент изменения ненадёжен на малых числах"))
no_drop = (m_ok["group"].isna() & (m_ok["chg"] > -0.3)).sum()
excl.append(("нет падения (chg > −30%)", no_drop, "кампания по возврату активности им не нужна"))
other_drop = (m_ok["group"].isna() & (m_ok["chg"] <= -0.3)).sum()
excl.append(("падение есть, но не массовый сегмент и не развлечения", other_drop,
             "разрозненные случаи без общей причины: под них нельзя сделать одну коммуникацию"))
no_consent = (m_ok["group"].notna() & ~m_ok["marketing_opt_in"]).sum()
excl.append(("подходят под группу, но нет согласия на коммуникацию", no_consent, "коммуникация запрещена"))
excl_df = pd.DataFrame(excl, columns=["кто", "клиентов", "почему не берём"])
excl_df

Отдельно: **переход с пластика на токен** (Pay-сервисы) — не падение, клиенты платят той же картой.
Метрика считает все каналы, поэтому такие клиенты в группы не попадают. Провал канала «Онлайн-банк»
в апреле–мае 2025 года — технический, а не поведенческий, кампанию под него не делаем.

## 8. Что проверили и что не подтвердилось

Гипотезы, которые вы проверили и отвергли. По одной строке на гипотезу,
с числом.

In [ ]:
mass = m_ok["segment"] == "Массовый"
kz_all = m_ok.groupby(m_ok["city"] == "Казань")["chg"].median()
kz_mass = m_ok[mass].groupby(m_ok.loc[mass, "city"] == "Казань")["chg"].median()
inc_all = m_ok.groupby("income_source")["chg"].median()
inc_mass = m_ok[mass].groupby("income_source")["chg"].median()
sat = m_ok.groupby("satisfaction_score")["chg"].median()
ob = txn_ok[txn_ok["channel"] == "Онлайн-банк"].groupby("month").size()

hyp = pd.DataFrame([
    ("Казань теряет активность сильнее других городов",
     f"медиана chg: Казань {kz_all[True]:.0%} против {kz_all[False]:.0%}; но внутри массового сегмента "
     f"{kz_mass[True]:.0%} против {kz_mass[False]:.0%}",
     "не подтвердилось: эффект состава (87% массовых в Казани)"),
    ("Зарплата в другом банке → падение",
     f"все: «другой банк» {inc_all['Зарплата в другом банке']:.0%}, «наш банк» {inc_all['Зарплата в нашем банке']:.0%}; "
     f"массовый: {inc_mass['Зарплата в другом банке']:.0%} против {inc_mass['Зарплата в нашем банке']:.0%}",
     "не подтвердилось: различие объясняется сегментом"),
    ("Недовольные (оценка 1–2) снижают активность сильнее",
     f"медиана chg по оценкам 1…5: {', '.join(f'{v:.0%}' for v in sat.values)} (n ответивших = {m_ok['satisfaction_score'].notna().sum()})",
     "слабо: различие только у оценки 1, опрос прошли 41%"),
    ("Клиенты перестали платить картой",
     f"операции «Карта» {index_by('channel').loc[MONTHS[12:], 'Карта'].mean():.2f}, "
     f"«Токен» {index_by('channel').loc[MONTHS[12:], 'Токен'].mean():.2f} от уровня 2024",
     "не подтвердилось: смена способа оплаты, а не отказ"),
    ("Провал «Онлайн-банка» в апреле–мае — уход клиентов",
     f"операций в месяц: март {ob[pd.Period('2025-03', 'M')]}, апрель {ob[pd.Period('2025-04', 'M')]}, "
     f"май {ob[pd.Period('2025-05', 'M')]}, июнь {ob[pd.Period('2025-06', 'M')]}",
     "не подтвердилось: восстановление в июне, похоже на сбой"),
], columns=["гипотеза", "число", "вывод"])
hyp

## 9. Сохранение результатов

Три файла: очищенная таблица для второго ноутбука, журнал изменений,
итоговый список кампании.

In [ ]:
# Очищенные успешные операции действующих клиентов в периоде — для второго ноутбука
clean = txn_ok.copy()
clean["month"] = clean["month"].astype(str)
clean.to_parquet("clean_data.parquet", index=False)

# Метрики по клиентам и группы — тоже для отчёта
out = m_ok[["group"]].reindex(metrics.index)
client_metrics = metrics.join(out).reset_index()
client_metrics["card_open_date"] = client_metrics["card_open_date"].astype(str)
client_metrics.to_parquet("client_metrics.parquet", index=False)

cleaning_log.to_csv("cleaning_log.csv", index=False, encoding="utf-8")
campaign[["client_id", "group"]].to_csv("campaign_list.csv", index=False, encoding="utf-8")
hyp.to_csv("hypotheses.csv", index=False, encoding="utf-8")
excl_df.to_csv("exclusions.csv", index=False, encoding="utf-8")
print("сохранено:", len(clean), "операций;", len(client_metrics), "клиентов;",
      len(cleaning_log), "решений в журнале;", len(campaign), "клиентов в кампании")